## 1. Import Data

In [6]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path(r"C:\Users\HP\Documents\PycharmProjects\Krittim.sys\lumino-vch3-data\csv")  # or Path("lumino-vch3-data/csv")

data_dict = pd.read_csv(DATA_DIR / "data_dictionary.csv")
items = pd.read_csv(DATA_DIR / "items.csv")
sets = pd.read_csv(DATA_DIR / "sets.csv")
receipts = pd.read_csv(DATA_DIR / "receipts.csv")
transfers = pd.read_csv(DATA_DIR / "transfers.csv")
contractor_moves = pd.read_csv(DATA_DIR / "contractor_moves.csv", low_memory=False)

for name, df in [("items", items), ("sets", sets), ("receipts", receipts), ("transfers", transfers), ("contractor_moves", contractor_moves)]:
    print(f"{name}: {df.shape} | cols: {list(df.columns)}")

items: (16, 5) | cols: ['item_code', 'item_name', 'unit', 'lead_time_days', 'role']
sets: (8, 8) | cols: ['set_no', 'parent_code', 'parent_name', 'parent_unit', 'child_code', 'child_name', 'child_unit', 'ratio_of_parent']
receipts: (2600, 7) | cols: ['date', 'grn_no', 'item_code', 'item_name', 'unit', 'godown', 'qty_received']
transfers: (1446, 10) | cols: ['date', 'voucher_no', 'voucher_type', 'direction', 'item_code', 'item_name', 'unit', 'godown_code', 'godown', 'qty']
contractor_moves: (231076, 7) | cols: ['date', 'type', 'item_code', 'item_name', 'unit', 'godown', 'qty']


In [7]:
print("transfers direction:", transfers["direction"].unique())
print("transfers voucher_type:", transfers["voucher_type"].unique())
print("contractor_moves type:", contractor_moves["type"].unique())
print("godowns:", sorted(receipts["godown"].dropna().unique()))

transfers direction: <StringArray>
['IN', 'OUT']
Length: 2, dtype: str
transfers voucher_type: <StringArray>
[  'Receipt from another store',       'Issue to another store',
   'Issue to the closing store', 'Receipt at the closing store']
Length: 4, dtype: str
contractor_moves type: <StringArray>
['ISSUE', 'CONSUMPTION', 'RETURN']
Length: 3, dtype: str
godowns: ['KANPUR AURAIYA', 'KANPUR DEHAT', 'KANPUR ETAWAH', 'KANPUR FARRUKHABAD', 'KANPUR KANNAUJ', 'KANPUR NAGAR']


## 2. Checking the datasets

In [10]:
print("== Items ==")
print(items)
print("== Sets ==")
print(sets)

== Items ==
    item_code                                          item_name unit  \
0   MEH050071                                           EYE HOOK  NOS   
1   MLU020008                   25 SQMM ALUMINIUM LUGS RING TYPE  NOS   
2   MLU020013                             25 SQMM ALUMINIUM LUGS  NOS   
3   MNB010007    HOT DIP GALVANIZED HEX BOLTS & NUTS WITH WASHER   KG   
4   MNB010128                  HOT DIP GALVANIZED BOLTS AND NUTS   KG   
5   MNB010174          G.I. NUTS & BOLTS (M16X65 MM) WITH WASHER   MT   
6   MCA020009  LT AERIAL BUNCHED XLPE CABLE 3X95 SQ.MM+1X70 S...   KM   
7   MCA090005                                   SUSPENSION CLAMP  NOS   
8   MCO010007                       WEDGE CONNECTORS / PG CLAMPS  NOS   
9   MEH050075                                     DEAD END CLAMP  NOS   
10  MPO010014                            PCC POLE 8.5 MTR 180 KG  PCS   
11  MRP070022  1x25 mm2 Un-armoured XLPE insulated Al. LT Cab...  MTR   
12  MRP070059       Insulating Piercing

In [12]:
valid_items = set(items["item_code"].str.strip())

for name, df, qty_col, doc_col in [
    ("RECEIPTS", receipts, "qty_received", "grn_no"),
    ("TRANSFERS", transfers, "qty", "voucher_no"),
]:
    print(f"=== {name} ({len(df)} rows) ===")
    print("Missing values:\n", df.isna().sum()[df.isna().sum() > 0])

    dt = pd.to_datetime(df["date"], errors="coerce")
    print(f"Date range: {dt.min().date()} to {dt.max().date()} | Invalid dates: {dt.isna().sum()}")

    q = df[qty_col]
    print(f"{qty_col}: min={q.min()}, max={q.max()}, zeros={(q == 0).sum()}, negatives={(q < 0).sum()}")

    bad_codes = set(df["item_code"].dropna().str.strip()) - valid_items
    print(f"Unknown item_codes: {bad_codes}")
    print(f"Exact duplicate rows: {df.duplicated().sum()} | Duplicate {doc_col}+item: {df.duplicated(subset=[doc_col, 'item_code']).sum()}\n")

=== RECEIPTS (2600 rows) ===
Missing values:
 Series([], dtype: int64)
Date range: 2023-03-01 to 2026-07-30 | Invalid dates: 0
qty_received: min=0.0, max=114500.0, zeros=2, negatives=0
Unknown item_codes: set()
Exact duplicate rows: 0 | Duplicate grn_no+item: 0

=== TRANSFERS (1446 rows) ===
Missing values:
 Series([], dtype: int64)
Date range: 2023-05-31 to 2026-04-21 | Invalid dates: 0
qty: min=0.001, max=119335.0, zeros=0, negatives=0
Unknown item_codes: set()
Exact duplicate rows: 0 | Duplicate voucher_no+item: 8



In [13]:
print("=== TRANSFERS BALANCE CHECK ===")
print(transfers.groupby(["voucher_type", "direction"])["qty"].agg(["count", "sum"]))

=== TRANSFERS BALANCE CHECK ===
                                        count          sum
voucher_type                 direction                    
Issue to another store       OUT          702  1009972.900
Issue to the closing store   OUT           74   418089.000
Receipt at the closing store IN            38   316925.097
Receipt from another store   IN           632   928538.084


In [14]:
valid_items = set(items["item_code"].str.strip())

for name, df, qty_col, doc_col in [
    ("RECEIPTS", receipts, "qty_received", "grn_no"),
    ("TRANSFERS", transfers, "qty", "voucher_no"),
]:
    print(f"=== {name} ({len(df)} rows) ===")
    print("Missing values:\n", df.isna().sum()[df.isna().sum() > 0])

    dt = pd.to_datetime(df["date"], errors="coerce")
    print(f"Date range: {dt.min().date()} to {dt.max().date()} | Invalid dates: {dt.isna().sum()}")

    q = df[qty_col]
    print(f"{qty_col}: min={q.min()}, max={q.max()}, zeros={(q == 0).sum()}, negatives={(q < 0).sum()}")

    bad_codes = set(df["item_code"].dropna().str.strip()) - valid_items
    print(f"Unknown item_codes: {bad_codes}")
    print(f"Exact duplicate rows: {df.duplicated().sum()} | Duplicate {doc_col}+item: {df.duplicated(subset=[doc_col, 'item_code']).sum()}\n")

=== RECEIPTS (2600 rows) ===
Missing values:
 Series([], dtype: int64)
Date range: 2023-03-01 to 2026-07-30 | Invalid dates: 0
qty_received: min=0.0, max=114500.0, zeros=2, negatives=0
Unknown item_codes: set()
Exact duplicate rows: 0 | Duplicate grn_no+item: 0

=== TRANSFERS (1446 rows) ===
Missing values:
 Series([], dtype: int64)
Date range: 2023-05-31 to 2026-04-21 | Invalid dates: 0
qty: min=0.001, max=119335.0, zeros=0, negatives=0
Unknown item_codes: set()
Exact duplicate rows: 0 | Duplicate voucher_no+item: 8



In [15]:
print("=== ZERO-QTY RECEIPTS ===")
print(receipts[receipts["qty_received"] == 0])

print("\n=== TRANSFERS BALANCE CHECK ===")
print(transfers.groupby(["voucher_type", "direction"])["qty"].agg(["count", "sum"]))

print("\n=== CONTRACTOR MOVES (231,076 rows) ===")
print("Missing values:\n", contractor_moves.isna().sum()[contractor_moves.isna().sum() > 0])
dt_cm = pd.to_datetime(contractor_moves["date"], errors="coerce")
print(f"Date range: {dt_cm.min().date()} to {dt_cm.max().date()} | Invalid dates: {dt_cm.isna().sum()}")
q_cm = contractor_moves["qty"]
print(f"qty: min={q_cm.min()}, max={q_cm.max()}, zeros={(q_cm == 0).sum()}, negatives={(q_cm < 0).sum()}")
print("Unknown item_codes:", set(contractor_moves["item_code"].dropna().str.strip()) - set(items["item_code"].str.strip()))
print("Godowns:", sorted(contractor_moves["godown"].dropna().str.strip().unique()))
print("\nBy Type:")
print(contractor_moves.groupby("type")["qty"].agg(["count", "sum", "min", "max"]))

=== ZERO-QTY RECEIPTS ===
            date       grn_no  item_code  \
199   2023-05-23  GS23523-002  MLU020008   
1332  2024-02-19  GS23219-002  MCA020009   

                                              item_name unit          godown  \
199                    25 SQMM ALUMINIUM LUGS RING TYPE  NOS   KANPUR ETAWAH   
1332  LT AERIAL BUNCHED XLPE CABLE 3X95 SQ.MM+1X70 S...   KM  KANPUR AURAIYA   

      qty_received  
199            0.0  
1332           0.0  

=== TRANSFERS BALANCE CHECK ===
                                        count          sum
voucher_type                 direction                    
Issue to another store       OUT          702  1009972.900
Issue to the closing store   OUT           74   418089.000
Receipt at the closing store IN            38   316925.097
Receipt from another store   IN           632   928538.084

=== CONTRACTOR MOVES (231,076 rows) ===
Missing values:
 godown    100650
dtype: int64
Date range: 2023-04-10 to 2026-10-29 | Invalid dates: 0
qty: m

In [18]:
OUT = Path("../backend_ready")
inv = pd.read_csv(OUT / "inventory_summary.csv")
rec = pd.read_csv(OUT / "receipts.csv")
tr = pd.read_csv(OUT / "transfers.csv")
cm = pd.read_csv(OUT / "contractor_movements.csv", low_memory=False)

print("=== 1. NEGATIVE STOCK CHECK ===")
neg = inv[inv["raw_stock"] < -1e-6]
print(f"Negative raw_stock rows: {len(neg)} out of {len(inv)}")
if len(neg) > 0:
    print(neg.groupby(["godown", "item_code"])["raw_stock"].min().head(10))

print("\n=== 2. EXACT DUPLICATE ROWS ===")
for name, df in [("receipts", rec), ("transfers", tr), ("contractor_movements", cm)]:
    print(f"{name}: {df.duplicated().sum()} exact duplicate rows")

print("\n=== 3. GODOWN & ITEM CONSISTENCY ===")
for name, df in [("receipts", rec), ("transfers", tr), ("contractor_movements", cm)]:
    print(f"{name} godowns:", sorted(df["godown"].dropna().unique()))

print("\n=== 4. EXTREME OUTLIER QUANTITIES (99.9th percentile vs max) ===")
for name, df, col in [("receipts", rec, "qty_received"), ("transfers", tr, "qty"), ("contractor_movements", cm, "qty")]:
    q999 = df[col].quantile(0.999)
    mx = df[col].max()
    print(f"{name} ({col}): 99.9%={q999:.2f}, max={mx:.2f}, count_above_999={(df[col] > q999).sum()}")

=== 1. NEGATIVE STOCK CHECK ===
Negative raw_stock rows: 21 out of 4224
godown              item_code
KANPUR AURAIYA      MCO010007     -3.000
                    MLU020008   -130.000
                    MNB010174     -0.025
                    MRP070065     -3.000
KANPUR FARRUKHABAD  MCO010007    -38.000
                    MEH050071    -49.000
                    MRP070060    -88.000
KANPUR KANNAUJ      MCA090005     -8.000
                    MRP070065     -1.000
Name: raw_stock, dtype: float64

=== 2. EXACT DUPLICATE ROWS ===
receipts: 690 exact duplicate rows
transfers: 43 exact duplicate rows
contractor_movements: 43783 exact duplicate rows

=== 3. GODOWN & ITEM CONSISTENCY ===
receipts godowns: ['KANPUR AURAIYA', 'KANPUR DEHAT', 'KANPUR ETAWAH', 'KANPUR FARRUKHABAD', 'KANPUR KANNAUJ', 'KANPUR NAGAR']
transfers godowns: ['KANPUR AURAIYA', 'KANPUR CLOSER', 'KANPUR DEHAT', 'KANPUR ETAWAH', 'KANPUR FARRUKHABAD', 'KANPUR KANNAUJ', 'KANPUR NAGAR']
contractor_movements godowns: ['KANPU